In [0]:
%sql

-- Preview the dim_date dimension table (one row per date: date_sk, year, quarter, month_name)
SELECT * FROM it_support_tickets.dbt_hdalavai.dim_date
    


In [0]:
%sql
-- Preview the dim_employee dimension table (employee keys and details)
SELECT * FROM it_support_tickets.dbt_hdalavai.dim_employee
    

In [0]:
%sql
-- Preview the dim_location dimension table (location keys and cities)
SELECT * FROM it_support_tickets.dbt_hdalavai.dim_location

In [0]:
%sql
-- Preview the dim_support dimension table (ticket attributes: category, priority, engineer, status)
SELECT * FROM it_support_tickets.dbt_hdalavai.dim_support

In [0]:
%sql
-- Preview the fact_support_tickets fact table (measures: resolution_hours, hourly_cost, total_resolution_cost, plus dimension keys)
select * from it_support_tickets.dbt_hdalavai.fact_support_tickets

In [0]:
%sql
-- Gold star-schema query: rebuild full ticket details by joining the fact table to all four dimensions
-- (dim_date has no month column, so month is derived with MONTH(created_date))
SELECT

    f.ticket_id,

    d.created_date,
    d.year,
    d.quarter,
    MONTH(d.created_date) AS month,
    d.month_name,

    e.employee_id,

    l.location,

    s.support_category,
    s.priority,
    s.support_engineer,
    s.ticket_status,

    f.resolution_hours,
    f.hourly_cost,
    f.total_resolution_cost,
    f.batch_id

FROM it_support_tickets.dbt_hdalavai.fact_support_tickets f

LEFT JOIN it_support_tickets.dbt_hdalavai.dim_date d
    ON f.date_sk = d.date_sk

LEFT JOIN it_support_tickets.dbt_hdalavai.dim_employee e
    ON f.employee_sk = e.employee_sk

LEFT JOIN it_support_tickets.dbt_hdalavai.dim_location l
    ON f.location_sk = l.location_sk

LEFT JOIN it_support_tickets.dbt_hdalavai.dim_support s
    ON f.support_sk = s.support_sk

ORDER BY f.ticket_id;

In [0]:
%sql
-- Total support cost per employee: join the fact table to employee/location/support dims, sum total_resolution_cost, highest first
select
    e.employee_id,
    s.support_engineer,
    l.location,
    sum(f.total_resolution_cost) as total_cost
from it_support_tickets.dbt_hdalavai.fact_support_tickets f
join it_support_tickets.dbt_hdalavai.dim_employee e
    on f.employee_sk = e.employee_sk
join it_support_tickets.dbt_hdalavai.dim_location l
    on f.location_sk = l.location_sk
join it_support_tickets.dbt_hdalavai.dim_support s
    on f.support_sk = s.support_sk
group by
    e.employee_id,
    s.support_engineer,
    l.location
order by total_cost desc;

In [0]:
%sql
-- Total resolution hours per support category, highest first
select
    s.support_category,
    sum(f.resolution_hours) as total_hours
from it_support_tickets.dbt_hdalavai.fact_support_tickets f
join it_support_tickets.dbt_hdalavai.dim_support s
    on f.support_sk = s.support_sk
group by
    s.support_category
order by
    total_hours desc;

In [0]:
# Category-level KPIs: total/avg resolution hours, avg hourly cost, and total/avg resolution cost per support category
select
    s.support_category,
    sum(f.resolution_hours) as total_resolution_hours,
    avg(f.hourly_cost) as avg_hourly_cost,
    sum(f.total_resolution_cost) as total_cost,
    avg(f.total_resolution_cost) as avg_total_cost
from it_support_tickets.dbt_hdalavai.fact_support_tickets f
join it_support_tickets.dbt_hdalavai.dim_support s
    on f.support_sk = s.support_sk
group by
    s.support_category
order by
    total_cost desc;